In [1]:
import pandas as pd
import numpy as np
import clickhouse_connect
from datetime import datetime, timedelta, timezone

df_lyca_calls = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/CEIR/clean_dumps/lyca_90_day_call_locations.csv",
    dtype={"calling_location": "string"},
    low_memory=False
)

df_airtel_calls = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/CEIR/clean_dumps/airtel_90_day_call_locations.csv",
    dtype={"calling_location": "string"},
    low_memory=False
)

df_mtn_calls = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/CEIR/clean_dumps/mtn_90_day_call_locations.csv",
    dtype={"calling_location": "string"},
    low_memory=False
)


In [2]:
df_airtel_calls.head()

,day,calling_number,calling_location
0,2025-10-01,256200100100,641010263416116
1,2025-10-02,256200100100,641010263435687
2,2025-10-03,256200100100,641010263416116
3,2025-10-23,256200100100,641010240110818
4,2025-10-11,256200900001,641010008712704


In [3]:
df_mtn_calls.head()

,day,calling_number,calling_location
0,2025-10-01,256391004067,641100012123412
1,2025-10-02,256391004067,641100012122582
2,2025-10-03,256391004067,641100012122583
3,2025-10-06,256391004067,641100012123412
4,2025-10-07,256391004067,641100012123412


In [4]:
df_mtn_calls.dtypes

day                    str
calling_number       int64
calling_location    string
dtype: object

#### Import KYC data - use it later to estimate airtel devices based on subscriber - device ratio

In [5]:
#Load KYC Data
df_NID = pd.read_pickle("/Users/wmuheki/Documents/Projects/Analytics/KYC/clean_dumps/df_NID_12_25.pkl") # National ID Subset


In [6]:
import clickhouse_connect

# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir',
    settings={
        'max_execution_time': 90,
        'max_memory_usage': 2_000_000_000,  # 2GB
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the gsma_devices table (correct schema)
gsma_query = """
SELECT
    tac,
    oem                    AS manufacturer,
    brand                  AS brand,
    model                  AS model_name,
    if(
        marketing_name = '' OR isNull(marketing_name),
        concat(brand, ' ', model),
        marketing_name
    )                       AS device_name,
    device_type,
    os_family              AS operating_system,
    os_version,
    year_released,
    has_2g,
    has_3g,
    has_4g,
    has_5g,
    sim_slots
FROM gsma_devices
"""
gsma_df = client.query_df(gsma_query)


In [7]:
len(gsma_df)

290402

In [8]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_execution_time': 120,
        'max_memory_usage': 2000000000,  # 2GB max
        'max_threads': 2,
        'priority': 5
    }
)

# --------------------------------------------------------------------------------

def fetch_in_chunks(
    client,
    table: str,
    cols: list[str],
    days: int = 90,
    chunk_size: int = 300_000,
    max_rows: int | None = None,
    order_key: str = "last_seen",
    tie_key: str = "msisdn",
):
    """
    Keyset pagination: ORDER BY last_seen DESC, msisdn DESC
    Next chunk uses (last_seen < cursor_last_seen) OR (last_seen = cursor_last_seen AND msisdn < cursor_msisdn)
    """
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)

    cursor_last_seen = datetime.now(timezone.utc)
    cursor_tie = ""  # msisdn (string)

    parts = []
    total = 0

    # always include order_key for cursor advancement
    select_cols = cols.copy()
    if order_key not in select_cols:
        select_cols.append(order_key)

    select_list = ",\n        ".join(select_cols)

    while True:
        q = f"""
        SELECT
            {select_list}
        FROM {table}
        WHERE {order_key} >= toDateTime('{cutoff.strftime("%Y-%m-%d %H:%M:%S")}')
          AND (
                {order_key} < toDateTime('{cursor_last_seen.strftime("%Y-%m-%d %H:%M:%S")}')
                OR ({order_key} = toDateTime('{cursor_last_seen.strftime("%Y-%m-%d %H:%M:%S")}') AND {tie_key} < '{cursor_tie}')
              )
        ORDER BY {order_key} DESC, {tie_key} DESC
        LIMIT {chunk_size}
        """

        df = client.query_df(q)
        if df.empty:
            break

        total += len(df)

        # keep only requested columns (drop cursor column if it wasn't requested)
        out = df[cols].copy()
        parts.append(out)

        # advance cursor using last row
        last = df.iloc[-1]
        cursor_last_seen = pd.to_datetime(last[order_key]).to_pydatetime()
        cursor_tie = str(last[tie_key])

        print(f"{table}: fetched {len(df):,} (total {total:,}) cursor={cursor_last_seen} / {cursor_tie}")

        if max_rows is not None and total >= max_rows:
            break

    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=cols)


# Step 2: Fetch data from the domestic_subscribers table
domestic_df_gold = fetch_in_chunks(
    client,
    table="domestic_subscribers",
    cols=["msisdn", "imsi", "imei"],
    days=100,
    chunk_size=500_000,     # tune up/down
    max_rows=60_000_000
)

# Step 3: Fetch data from the roamers table
roam_df_gold = fetch_in_chunks(
    client,
    table="roamers",
    cols=["msisdn", "imsi", "imei"],
    days=100,
    chunk_size=500_000,
    max_rows=4_500_000
)


domestic_subscribers: fetched 500,000 (total 500,000) cursor=2026-02-02 15:15:15 / 256764456948
domestic_subscribers: fetched 500,000 (total 1,000,000) cursor=2026-02-02 15:11:37 / 256772479070
domestic_subscribers: fetched 500,000 (total 1,500,000) cursor=2026-02-02 15:08:29 / 256778889691
domestic_subscribers: fetched 500,000 (total 2,000,000) cursor=2026-02-02 15:06:02 / 256763885968
domestic_subscribers: fetched 500,000 (total 2,500,000) cursor=2026-02-02 15:03:42 / 256771465656
domestic_subscribers: fetched 500,000 (total 3,000,000) cursor=2026-02-02 15:01:11 / 256786709924
domestic_subscribers: fetched 500,000 (total 3,500,000) cursor=2026-02-02 14:58:31 / 256767022545
domestic_subscribers: fetched 500,000 (total 4,000,000) cursor=2026-02-02 14:55:36 / 256786005479
domestic_subscribers: fetched 500,000 (total 4,500,000) cursor=2026-02-02 14:52:33 / 256766911669
domestic_subscribers: fetched 500,000 (total 5,000,000) cursor=2026-02-02 14:49:18 / 256791879518
domestic_subscribers: 

#### Import data extracted from CEIR Bronze Layer

In [9]:
"""
This data includes only unique entries for the past 90 days.
>> MSISDN, IMSI and IMEI for domestic subscribers
>> IMSI and IMEI for roam subscribers - since MTN currently doesn't provide MSISDN for roamers
"""

# --------------------------------------------------------------------------------
domestic_bronze = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/CEIR/dumps/domestic_subscribers_03022026.csv",
    dtype={
        "msisdn": "string",
        "imsi": "string",
        "imei": "string",
    },
    low_memory=False
)

# --------------------------------------------------------------------------------
roam_bronze = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/CEIR/dumps/roaming_subscribers_03022026.csv",
    dtype={
        "imsi": "string",
        "imei": "string",
    },
    low_memory=False
)


In [10]:
len(domestic_bronze)

203393253

In [11]:
len(roam_bronze)

7824036

#### Import Cell/ Tower information

In [12]:
df_mtn_cells = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/CEIR/dumps/MTN_Cell_File_UCC_29012026.csv",
    encoding="latin1",
    low_memory=False
)

df_airtel_cells = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/CEIR/dumps/Airtel_CellFile_UCC_29012026_clean.csv",
    encoding="latin1",
    low_memory=False
)

In [13]:
# View file structure
df_mtn_cells.head()

,LAC,CellID,Latitude,Longitude,azimuth,Technology,SubCounty,District,Site_ID,Site_Name
0,6003.0,641100600300030,0.300131,32.576617,191.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe
1,6003.0,641100600300031,0.300131,32.576617,280.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe
2,6003.0,641100600300032,0.300131,32.576617,74.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe
3,6003.0,641100600300033,0.300131,32.576617,191.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe
4,6003.0,641100600300035,0.300131,32.576617,280.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe


In [14]:
# Transform Cell dataframes
df_mtn_cells["MNO"] = "MTN"
df_airtel_cells["MNO"] = "AIRTEL"
df_mtn_cells = df_mtn_cells.rename(columns={"Site_Id": "Site_ID"})
df_airtel_cells = df_airtel_cells.rename(columns={"Site_Id": "Site_ID"})
print(df_mtn_cells.columns)
print(df_airtel_cells.columns)

Index(['LAC', 'CellID', 'Latitude', 'Longitude', 'azimuth', 'Technology',
       'SubCounty', 'District', 'Site_ID', 'Site_Name', 'MNO'],
      dtype='str')
Index(['LAC', 'CellID', 'Latitude', 'Longitude', 'azimuth', 'Technology',
       'SubCounty', 'District', 'Site_ID', 'Site_Name', 'MNO'],
      dtype='str')


In [15]:
# merge the cell data frames into one
df_cells = pd.concat([df_mtn_cells, df_airtel_cells], ignore_index=True)

In [16]:
# view outcome
print(df_cells.shape)
print(df_cells["MNO"].value_counts())
df_cells.head()

(166455, 11)
MNO
MTN       103277
AIRTEL     63178
Name: count, dtype: int64


,LAC,CellID,Latitude,Longitude,azimuth,Technology,SubCounty,District,Site_ID,Site_Name,MNO
0,6003.0,641100600300030,0.300131,32.576617,191.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe,MTN
1,6003.0,641100600300031,0.300131,32.576617,280.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe,MTN
2,6003.0,641100600300032,0.300131,32.576617,74.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe,MTN
3,6003.0,641100600300033,0.300131,32.576617,191.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe,MTN
4,6003.0,641100600300035,0.300131,32.576617,280.0,3G,MAKINDYE,KAMPALA,3,BMK_Katwe,MTN


#### Transform Calling Data

In [17]:
# transform and merge all the calling data frames into one
for df in (df_airtel_calls, df_mtn_calls, df_lyca_calls):
    df["calling_number"] = df["calling_number"].astype("string")


df_airtel_calls["MNO"] = "AIRTEL"
df_mtn_calls["MNO"]    = "MTN"
df_lyca_calls["MNO"]   = "LYCA"

for df in (df_airtel_calls, df_mtn_calls, df_lyca_calls):
    df["MNO"] = df["MNO"].str.upper()

df_calls = pd.concat(
    [df_airtel_calls, df_mtn_calls, df_lyca_calls],
    ignore_index=True
)

In [18]:
df_calls["MNO"].value_counts()

MNO
MTN       59144797
AIRTEL    59034914
LYCA       2200480
Name: count, dtype: int64

In [19]:
df_calls.head()

,day,calling_number,calling_location,MNO
0,2025-10-01,256200100100,641010263416116,AIRTEL
1,2025-10-02,256200100100,641010263435687,AIRTEL
2,2025-10-03,256200100100,641010263416116,AIRTEL
3,2025-10-23,256200100100,641010240110818,AIRTEL
4,2025-10-11,256200900001,641010008712704,AIRTEL


#### Enrich calling data with cell/ tower data (left merge)

In [20]:
cell_cols = ["CellID", "Latitude", "Longitude", "Technology", "District"]

df_calls["calling_location"] = df_calls["calling_location"].astype("string")
df_cells["CellID"] = df_cells["CellID"].astype("string")

cells_map = (
    df_cells[cell_cols]
    .drop_duplicates(subset=["CellID"])
)

df_calls_cells = df_calls.merge(
    cells_map,
    how="left",
    left_on="calling_location",
    right_on="CellID"
)

df_calls_cells = df_calls_cells.drop(columns=["CellID"])

# row count must remain unchanged
assert len(df_calls_cells) == len(df_calls)

# cell mapping coverage
df_calls_cells["District"].notna().mean()


np.float64(0.884007037337231)

In [21]:
# Preview
df_calls_cells.head()

,day,calling_number,calling_location,MNO,Latitude,Longitude,Technology,District
0,2025-10-01,256200100100,641010263416116,AIRTEL,0.387600,32.59325,3G,Kampala
1,2025-10-02,256200100100,641010263435687,AIRTEL,NaN,NaN,NaN,NaN
2,2025-10-03,256200100100,641010263416116,AIRTEL,0.387600,32.59325,3G,Kampala
3,2025-10-23,256200100100,641010240110818,AIRTEL,NaN,NaN,NaN,NaN
4,2025-10-11,256200900001,641010008712704,AIRTEL,NaN,NaN,NaN,NaN


In [22]:
# check for null districts
df_calls_cells.groupby("MNO")["District"].apply(lambda s: s.isna().sum())

MNO
AIRTEL    11423636
LYCA       2200476
MTN         339143
Name: District, dtype: int64

#### Generate a cleaning data frame of calling data - Take most frequent district for each call log

In [23]:
df = df_calls_cells.copy()

df["calling_number"] = df["calling_number"].astype("string")
df["District"] = df["District"].astype("string")
df["day"] = pd.to_datetime(df["day"], errors="coerce")

# 1) mode district per calling_number (ignore missing District)
mode_district = (
    df.dropna(subset=["District"])
      .groupby("calling_number")["District"]
      .agg(lambda s: s.value_counts().idxmax())
)

df["mode_district"] = df["calling_number"].map(mode_district)

# 2) rank rows per calling_number
is_mode = df["District"].eq(df["mode_district"]).fillna(False)   # <- key fix
has_dist = df["District"].notna().fillna(False)

df["_rank"] = is_mode.astype("int8") * 2 + has_dist.astype("int8")

df_one_call = (
    df.sort_values(["calling_number", "_rank", "day"], ascending=[True, False, False])
      .drop_duplicates(subset=["calling_number"], keep="first")
      .drop(columns=["_rank", "mode_district"])
)

df_one_call.head()

,day,calling_number,calling_location,MNO,Latitude,Longitude,Technology,District
2,2025-10-03,256200100100,641010263416116,AIRTEL,0.387600,32.593250,3G,Kampala
31,2025-11-16,256200900001,641010401413952,AIRTEL,0.412777,32.557777,2G,Wakiso
43,2025-11-11,256200900002,641010202616122,AIRTEL,0.358827,32.672799,3G,Wakiso
58,2025-11-03,256200900003,641010032812304,AIRTEL,NaN,NaN,NaN,<NA>
81,2025-11-07,256200900006,641010007236944,AIRTEL,NaN,NaN,NaN,<NA>


In [24]:
# Sanity Checks

# should be 1 row per calling_number
df_one_call["calling_number"].duplicated().sum()

# compare counts
print("before rows:", len(df))
print("after rows :", len(df_one_call))
print("unique numbers:", df["calling_number"].nunique(), df_one_call["calling_number"].nunique())

before rows: 120380191
after rows : 6230104
unique numbers: 6230104 6230104


#### We enrich the calling data with IMEI information from the domestic subscriber data (CEIR gold layer)

In [25]:
"""
Check if any MSISDN has multiple IMEIs
though the CEIR dataset should have one IMEI per MSISDN since latest IMEI is stored in Gold Layer
"""
multi_imei = (
    domestic_df_gold
    .groupby("msisdn")["imei"]
    .nunique()
    .gt(1)
    .sum()
)

print("MSISDNs with >1 IMEI:", multi_imei)

MSISDNs with >1 IMEI: 0


In [26]:
dom_map = (
    domestic_df_gold[["msisdn", "imei"]]
    .astype({"msisdn": "string"})
    .dropna(subset=["msisdn", "imei"])
    .drop_duplicates(subset=["msisdn"], keep="first")
)

df_calls_enriched = df_one_call.merge(
    dom_map,
    how="left",
    left_on="calling_number",
    right_on="msisdn"
).drop(columns=["msisdn"])

In [27]:
# Check total records remaining
len(df_calls_enriched)

6230104

In [28]:
# preview
df_calls_enriched.head()

,day,calling_number,calling_location,MNO,Latitude,Longitude,Technology,District,imei
0,2025-10-03,256200100100,641010263416116,AIRTEL,0.387600,32.593250,3G,Kampala,35355480847878
1,2025-11-16,256200900001,641010401413952,AIRTEL,0.412777,32.557777,2G,Wakiso,35203077458772
2,2025-11-11,256200900002,641010202616122,AIRTEL,0.358827,32.672799,3G,Wakiso,35195208530990
3,2025-11-03,256200900003,641010032812304,AIRTEL,NaN,NaN,NaN,<NA>,<NA>
4,2025-11-07,256200900006,641010007236944,AIRTEL,NaN,NaN,NaN,<NA>,35781608223964


In [29]:
# check IMEI coverage
df_calls_enriched["imei"].notna().mean()

np.float64(0.8722241233854202)

In [30]:
# check IMEI coverage by operator
df_calls_enriched.groupby("MNO")["imei"].agg(lambda s: s.notna().mean())

MNO
AIRTEL    0.846326
LYCA      0.000231
MTN       0.970906
Name: imei, dtype: float64

In [31]:
# Check Null IMEI Coverage by Operator
df_calls_enriched.groupby("MNO")["imei"].apply(lambda s: s.isna().sum())

MNO
AIRTEL    457927
LYCA      250867
MTN        87263
Name: imei, dtype: int64

#### Enrich all data sets with GSMA data

In [32]:

# ensure string
df_calls_enriched["imei"] = df_calls_enriched["imei"].astype("string")
domestic_df_gold["imei"]= domestic_df_gold["imei"].astype("string")
roam_df_gold["imei"]= roam_df_gold["imei"].astype("string")

# derive TAC for all data sets
df_calls_enriched["tac"] = (
    df_calls_enriched["imei"]
    .where(df_calls_enriched["imei"].str.fullmatch(r"\d{8,}"))
    .str.slice(0, 8)
)

domestic_df_gold["tac"] = (
    domestic_df_gold["imei"]
    .where(domestic_df_gold["imei"].str.fullmatch(r"\d{8,}"))
    .str.slice(0, 8)
)

roam_df_gold["tac"] = (
    roam_df_gold["imei"]
    .where(roam_df_gold["imei"].str.fullmatch(r"\d{8,}"))
    .str.slice(0, 8)
)


domestic_bronze["tac"] = (
    domestic_bronze["imei"]
    .where(domestic_bronze["imei"].str.fullmatch(r"\d{8,}"))
    .str.slice(0, 8)
)

roam_bronze["tac"] = (
    roam_bronze["imei"]
    .where(roam_bronze["imei"].str.fullmatch(r"\d{8,}"))
    .str.slice(0, 8)
)


# Prepare GSMA Table
gsma_df["tac"] = gsma_df["tac"].astype("string")

gsma_map = (
    gsma_df
    .drop_duplicates(subset=["tac"])
)

# Left Join
df_calling_devices = df_calls_enriched.merge(
    gsma_map,
    how="left",
    on="tac"
)

df_domestic_devices_gold = domestic_df_gold.merge(
    gsma_map,
    how="left",
    on="tac"
)

df_roaming_devices_gold = roam_df_gold.merge(
    gsma_map,
    how="left",
    on="tac"
)

df_domestic_devices_bronze = domestic_bronze.merge(
    gsma_map,
    how="left",
    on="tac"
)

df_roaming_devices_bronze = roam_bronze.merge(
    gsma_map,
    how="left",
    on="tac"
)

# Mark Valid and Fake Device IMEIs
cols = ["manufacturer", "device_name", "device_type", "model_name"]

df_calling_devices["tac_status"] = np.where(
    df_calling_devices[cols].notna().any(axis=1),
    "VALID",
    "FAKE"
)

df_domestic_devices_gold["tac_status"] = np.where(
    df_domestic_devices_gold[cols].notna().any(axis=1),
    "VALID",
    "FAKE"
)

df_roaming_devices_gold["tac_status"] = np.where(
    df_roaming_devices_gold[cols].notna().any(axis=1),
    "VALID",
    "FAKE"
)


df_domestic_devices_bronze["tac_status"] = np.where(
    df_domestic_devices_bronze[cols].notna().any(axis=1),
    "VALID",
    "FAKE"
)

df_roaming_devices_bronze["tac_status"] = np.where(
    df_roaming_devices_bronze[cols].notna().any(axis=1),
    "VALID",
    "FAKE"
)

In [33]:
df_calling_devices.head()

,day,calling_number,calling_location,MNO,Latitude,Longitude,Technology,District,imei,tac,...,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status
0,2025-10-03,256200100100,641010263416116,AIRTEL,0.387600,32.593250,3G,Kampala,35355480847878,35355480,...,Smartphone,iOS,17,2023.0,1.0,1.0,1.0,1.0,1.0,VALID
1,2025-11-16,256200900001,641010401413952,AIRTEL,0.412777,32.557777,2G,Wakiso,35203077458772,35203077,...,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE
2,2025-11-11,256200900002,641010202616122,AIRTEL,0.358827,32.672799,3G,Wakiso,35195208530990,35195208,...,Smartphone,Android,6.0,2016.0,1.0,1.0,1.0,0.0,0.0,VALID
3,2025-11-03,256200900003,641010032812304,AIRTEL,NaN,NaN,NaN,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE
4,2025-11-07,256200900006,641010007236944,AIRTEL,NaN,NaN,NaN,<NA>,35781608223964,35781608,...,Smartphone,Android,7,2017.0,1.0,1.0,1.0,0.0,0.0,VALID


In [34]:
df_domestic_devices_gold.head()

,msisdn,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status
0,256793353631,641101970622258,35028262324305,35028262,Tecno Telecom (HK) Limited,TECNO,BF6,Pop 7,Smartphone,Android,12,2023.0,1.0,1.0,1.0,0.0,2.0,VALID
1,256793318569,641101968288166,35654139428011,35654139,"Motorola Mobility LLC, a Lenovo Company",Motorola,XT2255-1,Moto g72,Smartphone,Android,12,2022.0,1.0,1.0,1.0,0.0,2.0,VALID
2,256793310689,641101970116161,35961615424410,35961615,Samsung Korea,Samsung,SM-A055F/DS,Galaxy A05,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID
3,256792935092,641101968143959,35737059527223,35737059,Tecno Telecom (HK) Limited,TECNO,T528,TECNO T528,Mobile Phone/Feature phone,,,2017.0,1.0,0.0,0.0,0.0,2.0,VALID
4,256792833376,641101965613800,35594211180377,35594211,INFINIX TECHNOLOGY LIMITED,Infinix,X650B,Hot 8,Smartphone,Android,9,2019.0,1.0,1.0,1.0,0.0,2.0,VALID


In [35]:
df_roaming_devices_gold.head()

,msisdn,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status
0,,222013090790158,35975900251493,35975900,Not Known,Not Known,6020,6020,Handheld,Nokia OS,,2005.0,1.0,0.0,0.0,0.0,0.0,VALID
1,,222013090791558,35975900251493,35975900,Not Known,Not Known,6020,6020,Handheld,Nokia OS,,2005.0,1.0,0.0,0.0,0.0,0.0,VALID
2,,204047600873650,35975900251493,35975900,Not Known,Not Known,6020,6020,Handheld,Nokia OS,,2005.0,1.0,0.0,0.0,0.0,0.0,VALID
3,,901430004648253,35975900251493,35975900,Not Known,Not Known,6020,6020,Handheld,Nokia OS,,2005.0,1.0,0.0,0.0,0.0,0.0,VALID
4,211924056637,659020015013153,35464802000025,35464802,Not Known,Not Known,Bird S302,Not Known Bird S302,Handheld,Other,,2008.0,1.0,0.0,0.0,0.0,0.0,VALID


In [36]:
df_domestic_devices_bronze.head()

,msisdn,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status
0,256786027767,641101937027356,35708348938858,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID
1,256764669826,641101953740203,35708348938858,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID
2,256767136425,641101950628984,35708348938928,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID
3,256768288076,641101963244197,35708348938929,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID
4,256780203711,641101926585693,35708348938940,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID


In [37]:
df_roaming_devices_bronze.head()

,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status
0,639035116276891,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE
1,630021242405697,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE
2,635130148725358,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE
3,630021256083190,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE
4,639035032347035,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE


In [38]:
len(df_domestic_devices_bronze)

203393253

#### Enrich Roaming Data with Country of SIM Origin/ Registration - From IMSI

In [40]:
mcc_lu = pd.read_csv("/Users/wmuheki/Documents/Projects/Analytics/CEIR/clean_dumps/MCC_Each_country.csv", dtype=str)
mcc_lu.head()


,MCC,Country
0,289,Abkhazia
1,412,Afghanistan
2,276,Albania
3,603,Algeria
4,544,American Samoa


In [41]:

# 1) Normalize lookup columns
mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])

# 2) Extract MCC from IMSI (first 3 digits) - for both roaming data frames
df_roaming_devices_gold["imsi"] = df_roaming_devices_gold["imsi"].astype("string")
df_roaming_devices_bronze["imsi"] = df_roaming_devices_bronze["imsi"].astype("string")

df_roaming_devices_gold["mcc"] = (
    df_roaming_devices_gold["imsi"]
    .str.replace(r"\D+", "", regex=True)  # keep digits only
    .str.slice(0, 3)
)

df_roaming_devices_bronze["mcc"] = (
    df_roaming_devices_bronze["imsi"]
    .str.replace(r"\D+", "", regex=True)  # keep digits only
    .str.slice(0, 3)
)

# 3) Merge Country Code Data Frame with the Roaming Data sets
df_roaming_devices_gold = df_roaming_devices_gold.merge(
    mcc_lu[["mcc", "country"]],
    how="left",
    on="mcc"
)

df_roaming_devices_bronze = df_roaming_devices_bronze.merge(
    mcc_lu[["mcc", "country"]],
    how="left",
    on="mcc"
)

# 4) Optional: fill unknowns - for both data sets
df_roaming_devices_gold["country"] = df_roaming_devices_gold["country"].fillna("UNKNOWN")
df_roaming_devices_bronze["country"] = df_roaming_devices_bronze["country"].fillna("UNKNOWN")


In [42]:
df_roaming_devices_gold.head()

,msisdn,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status,mcc,country
0,,222013090790158,35975900251493,35975900,Not Known,Not Known,6020,6020,Handheld,Nokia OS,,2005.0,1.0,0.0,0.0,0.0,0.0,VALID,222,Italy
1,,222013090791558,35975900251493,35975900,Not Known,Not Known,6020,6020,Handheld,Nokia OS,,2005.0,1.0,0.0,0.0,0.0,0.0,VALID,222,Italy
2,,204047600873650,35975900251493,35975900,Not Known,Not Known,6020,6020,Handheld,Nokia OS,,2005.0,1.0,0.0,0.0,0.0,0.0,VALID,204,Netherlands
3,,901430004648253,35975900251493,35975900,Not Known,Not Known,6020,6020,Handheld,Nokia OS,,2005.0,1.0,0.0,0.0,0.0,0.0,VALID,901,International Networks
4,211924056637,659020015013153,35464802000025,35464802,Not Known,Not Known,Bird S302,Not Known Bird S302,Handheld,Other,,2008.0,1.0,0.0,0.0,0.0,0.0,VALID,659,South Sudan


In [43]:
df_roaming_devices_bronze.head()

,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status,mcc,country
0,639035116276891,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE,639,Kenya
1,630021242405697,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE,630,Democratic Republic of Congo
2,635130148725358,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE,635,Rwanda
3,630021256083190,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE,630,Democratic Republic of Congo
4,639035032347035,00000000000000,00000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,FAKE,639,Kenya


#### Enrich Domestic Data with MNO - based on IMSI

In [44]:

# Ensure IMSI is string (important)
df_domestic_devices_gold["imsi"] = df_domestic_devices_gold["imsi"].astype(str)
df_domestic_devices_bronze["imsi"] = df_domestic_devices_bronze["imsi"].astype(str)

# Derive MNO from IMSI prefix
df_domestic_devices_gold["mno"] = (
    df_domestic_devices_gold["imsi"]
    .str[:5]
    .map({
        "64110": "MTN",
        "64101": "AIRTEL",
        "64122": "AIRTEL",
        "64120": "HAMILTON",
        "64108": "TALKIO",
        "64114": "AFRICELL",
        "64104": "ORANGE",
    })
    .fillna("UNKNOWN")
)

df_domestic_devices_bronze["mno"] = (
    df_domestic_devices_bronze["imsi"]
    .str[:5]
    .map({
       "64110": "MTN",
        "64101": "AIRTEL",
        "64122": "AIRTEL",
        "64120": "HAMILTON",
        "64108": "TALKIO",
        "64114": "AFRICELL",
        "64104": "ORANGE",
    })
    .fillna("UNKNOWN")
)


In [45]:
# Preview
df_domestic_devices_gold.head()

,msisdn,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status,mno
0,256793353631,641101970622258,35028262324305,35028262,Tecno Telecom (HK) Limited,TECNO,BF6,Pop 7,Smartphone,Android,12,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN
1,256793318569,641101968288166,35654139428011,35654139,"Motorola Mobility LLC, a Lenovo Company",Motorola,XT2255-1,Moto g72,Smartphone,Android,12,2022.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN
2,256793310689,641101970116161,35961615424410,35961615,Samsung Korea,Samsung,SM-A055F/DS,Galaxy A05,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN
3,256792935092,641101968143959,35737059527223,35737059,Tecno Telecom (HK) Limited,TECNO,T528,TECNO T528,Mobile Phone/Feature phone,,,2017.0,1.0,0.0,0.0,0.0,2.0,VALID,MTN
4,256792833376,641101965613800,35594211180377,35594211,INFINIX TECHNOLOGY LIMITED,Infinix,X650B,Hot 8,Smartphone,Android,9,2019.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN


In [46]:
# Preview
df_domestic_devices_bronze.head()

,msisdn,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status,mno
0,256786027767,641101937027356,35708348938858,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN
1,256764669826,641101953740203,35708348938858,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN
2,256767136425,641101950628984,35708348938928,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN
3,256768288076,641101963244197,35708348938929,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN
4,256780203711,641101926585693,35708348938940,35708348,Itel Technology Limited,itel,A663L,A05s,Smartphone,Android,13,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN


In [47]:
# Sanity Checks
df_domestic_devices_gold["mno"].value_counts()

mno
MTN         24894408
AIRTEL      19383308
HAMILTON          30
TALKIO             1
Name: count, dtype: int64

In [48]:
# Sanity Checks
df_domestic_devices_bronze["mno"].value_counts()

mno
AIRTEL      120320698
MTN          83072452
HAMILTON           68
ORANGE             26
AFRICELL            6
TALKIO              2
UNKNOWN             1
Name: count, dtype: int64

In [49]:
df_domestic_devices_gold.loc[
    df_domestic_devices_gold["mno"] == "UNKNOWN",
    ["imsi", "msisdn", "imei", "tac", "device_name", "model_name"]
].head(35)

,imsi,msisdn,imei,tac,device_name,model_name


In [50]:
df_domestic_devices_bronze.loc[
    df_domestic_devices_bronze["mno"] == "UNKNOWN",
    ["imsi", "msisdn", "imei", "tac", "device_name", "model_name"]
].head(35)

,imsi,msisdn,imei,tac,device_name,model_name
180527774,641110105032455,<NA>,35464802000025,35464802,Not Known Bird S302,Bird S302


In [51]:
# We drop unknown records and those for TALKIO, ORANGE, HAMILTON and AFRICELL
DROP_MNOS = ["UNKNOWN", "ORANGE", "AFRICELL", "HAMILTON", "TALKIO"]

df_domestic_devices_gold = (
    df_domestic_devices_gold
    .loc[~df_domestic_devices_gold["mno"].isin(DROP_MNOS)]
    .reset_index(drop=True)
)

df_domestic_devices_bronze = (
    df_domestic_devices_bronze
    .loc[~df_domestic_devices_bronze["mno"].isin(DROP_MNOS)]
    .reset_index(drop=True)
)


###### On a later date - we need to investigate the source of 14, and 04 MNC codes since these MNOs exited the market

#### Enrich Domestic Data with Gender and Age

In [52]:
# Change all MSISDNs to start with 256
df_NID["msisdn"] = (
    df_NID["msisdn"]
    .astype("string")
    .str.strip()
    .str.replace(r"^0", "256", regex=True)
)

In [53]:
df_NID["msisdn"] = df_NID["msisdn"].astype(str)
df_domestic_devices_gold["msisdn"] = df_domestic_devices_gold["msisdn"].astype(str)
df_domestic_devices_bronze["msisdn"] = df_domestic_devices_bronze["msisdn"].astype(str)

nid_dim = (
    df_NID[["msisdn", "gender", "age"]]
    .drop_duplicates(subset=["msisdn"])
)


df_domestic_devices_gold = (
    df_domestic_devices_gold
    .merge(
        nid_dim,
        on="msisdn",
        how="left",
        validate="m:1"   # many device rows → one subscriber row
    )
)


df_domestic_devices_bronze = (
    df_domestic_devices_bronze
    .merge(
        nid_dim,
        on="msisdn",
        how="left",
        validate="m:1"   # many device rows → one subscriber row
    )
)


# Analysis

#### Smartphone Only

In [55]:
device_type_gold = (
    df_domestic_devices_gold
    .loc[df_domestic_devices_gold["device_type"].str.lower() == "smartphone"]
    .pivot_table(
        index="device_name",   # rows
        columns="mno",          # columns
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

# Row-wise total (across MNOs)
device_type_gold["Total"] = device_type_gold.sum(axis=1)

# Optional: sort by Total descending
device_type_gold = device_type_gold.sort_values("Total", ascending=False)

device_type_gold.head(100)

mno,AIRTEL,MTN,Total
device_name,,,
Galaxy A05,343484,608080,951564
Spark Go 2024,301621,442683,744304
Galaxy A06,165830,273257,439087
Pop 7,162156,218250,380406
Galaxy A03 Core,131744,177089,308833
...,...,...,...
Y93,12964,15192,28156
Pop 2,11565,16475,28040
Camon 40,11158,16758,27916


In [56]:
man_gold = (
    df_domestic_devices_gold
    .loc[df_domestic_devices_gold["device_type"].str.lower() == "smartphone"]
    .pivot_table(
        index="manufacturer",   # rows
        columns="mno",          # columns
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

# Row-wise total (across MNOs)
man_gold["Total"] = man_gold.sum(axis=1)

# Optional: sort by Total descending
man_gold = man_gold.sort_values("Total", ascending=False)

man_gold.head(50)

mno,AIRTEL,MTN,Total
manufacturer,,,
Tecno Telecom (HK) Limited,1513591,2408916,3922507
Samsung Korea,1178675,2386594,3565269
Itel Technology Limited,781242,1044483,1825725
INFINIX TECHNOLOGY LIMITED,652878,1037972,1690850
Not Known,171352,233403,404755
Apple Inc,95525,287085,382610
Xiaomi Communications Co Ltd,123180,196926,320106
Guangdong Oppo Mobile Telecommunications Corp Ltd,102151,147432,249583
Shenzhen Simi Electronic Co Ltd,129503,104675,234178


In [57]:
tac_gold = (
    df_domestic_devices_gold
    .loc[df_domestic_devices_gold["device_type"].str.lower() == "smartphone"]
    .pivot_table(
        index="mno",
        columns="tac_status",
        values="imei",          # any column; we're just counting rows
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

tac_gold["Total"] = tac_gold.sum(axis=1)

# optional: sort by Total desc
tac_gold = tac_gold.sort_values("Total", ascending=False)

display(tac_gold)

tac_status,VALID,Total
mno,,
MTN,8930264,8930264
AIRTEL,5521500,5521500


#### All Device Types

In [70]:
import pandas as pd

# ----------------------------
# Device counts by device_name, with device_type column
# ----------------------------

df = df_domestic_devices_gold.copy()

# normalize (avoids merge/pivot mismatches)
df["mno"] = df["mno"].astype(str).str.upper().str.strip()
df["device_name"] = df["device_name"].astype(str).str.strip()
df["device_type"] = df["device_type"].astype(str).str.strip()
df["imei"] = df["imei"].astype(str).str.strip()

# 1) Build a lookup: device_name -> device_type (choose most common if inconsistent)
device_type_dim = (
    df.dropna(subset=["device_name"])
      .groupby("device_name")["device_type"]
      .agg(lambda s: s.value_counts(dropna=False).index[0])
      .rename("device_type")
      .reset_index()
)

# 2) Pivot counts (use nunique to avoid duplicate IMEI inflation; change to 'size' if you want raw rows)
device_type_gold = (
    df.pivot_table(
        index="device_name",
        columns="mno",
        values="imei",
        aggfunc=pd.Series.nunique,   # <- recommended
        fill_value=0,
        dropna=False
    )
)

# 3) Add Total
device_type_gold["Total"] = device_type_gold.sum(axis=1)

# 4) Join device_type column back in
device_type_gold = (
    device_type_gold.reset_index()
    .merge(device_type_dim, on="device_name", how="left")
    .sort_values("Total", ascending=False)
)

# 5) Display
device_type_gold.head(100)

,device_name,AIRTEL,MTN,Total,device_type
33603,it2160,3480017,4289765,7769782,Mobile Phone/Feature phone
33690,itel it2163N,895792,793889,1689681,Mobile Phone/Feature phone
28706,TECNO T528,602459,972096,1574555,Mobile Phone/Feature phone
34896,NaN,679491,805570,1485061,NaN
28617,TECNO T101,564851,752450,1317301,Mobile Phone/Feature phone
...,...,...,...,...,...
28637,TECNO T313,22078,26900,48978,Mobile Phone/Feature phone
1846,Alola 4G,12808,36063,48871,Smartphone
3199,C22,20255,28362,48617,Smartphone
7187,Galaxy A14,11910,35525,47435,Smartphone


In [82]:
df_domestic_devices_gold.head()

,msisdn,imsi,imei,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,...,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots,tac_status,mno,gender,age
0,256793353631,641101970622258,35028262324305,35028262,Tecno Telecom (HK) Limited,TECNO,BF6,Pop 7,Smartphone,Android,...,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN,NaN,<NA>
1,256793318569,641101968288166,35654139428011,35654139,"Motorola Mobility LLC, a Lenovo Company",Motorola,XT2255-1,Moto g72,Smartphone,Android,...,2022.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN,NaN,<NA>
2,256793310689,641101970116161,35961615424410,35961615,Samsung Korea,Samsung,SM-A055F/DS,Galaxy A05,Smartphone,Android,...,2023.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN,NaN,<NA>
3,256792935092,641101968143959,35737059527223,35737059,Tecno Telecom (HK) Limited,TECNO,T528,TECNO T528,Mobile Phone/Feature phone,,...,2017.0,1.0,0.0,0.0,0.0,2.0,VALID,MTN,NaN,<NA>
4,256792833376,641101965613800,35594211180377,35594211,INFINIX TECHNOLOGY LIMITED,Infinix,X650B,Hot 8,Smartphone,Android,...,2019.0,1.0,1.0,1.0,0.0,2.0,VALID,MTN,NaN,<NA>


In [87]:

# ----------------------------
# Final table: manufacturer + device_name counts by MNO + Total
# + device_type
# + Technology (4G/5G only: 4G, 5G, 4G+5G, NONE)
# + SIM Slot (supported slots: 1, 2, 3+, UNKNOWN)
# ----------------------------

df = df_domestic_devices_gold.copy()

# ---- normalize (prevents pivot/merge mismatches)
df["mno"] = df["mno"].astype(str).str.upper().str.strip()
for c in ["manufacturer", "device_name", "device_type", "imei"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

# ---- normalize flags + sim slots
has4 = pd.to_numeric(df.get("has_4g"), errors="coerce").fillna(0).astype(int)
has5 = pd.to_numeric(df.get("has_5g"), errors="coerce").fillna(0).astype(int)
sim  = pd.to_numeric(df.get("sim_slots"), errors="coerce")

# ---- derive Technology (4G / 5G only)
df["Technology"] = "UNKNOWN"
df.loc[(has4 == 1) & (has5 == 0), "Technology"] = "4G"
df.loc[(has4 == 0) & (has5 == 1), "Technology"] = "5G"
df.loc[(has4 == 1) & (has5 == 1), "Technology"] = "4G+5G"
df.loc[(has4 == 0) & (has5 == 0), "Technology"] = "NONE"

# ---- derive SIM Slot category
df["SIM Slot"] = "UNKNOWN"
df.loc[sim == 1, "SIM Slot"] = "1"
df.loc[sim == 2, "SIM Slot"] = "2"
df.loc[sim >= 3, "SIM Slot"] = "3+"

# ---- helper: "most common value" per device_name
def mode_pick(s: pd.Series):
    s = s.dropna()
    if s.empty:
        return pd.NA
    return s.value_counts().index[0]

# ---- build device dimension (per device_name)
dim = (
    df.groupby("device_name")
      .agg(
          manufacturer=("manufacturer", mode_pick),
          device_type=("device_type", mode_pick),
          Technology=("Technology", mode_pick),
          **{"SIM Slot": ("SIM Slot", mode_pick)},
      )
      .reset_index()
)

# ---- pivot counts (unique IMEIs per MNO)
pt = df.pivot_table(
    index="device_name",
    columns="mno",
    values="imei",
    aggfunc=pd.Series.nunique,   # safer than size
    fill_value=0,
    dropna=False
)

pt["Total"] = pt.sum(axis=1)

# ---- final table (explicit column order)
final = (
    pt.reset_index()
      .merge(dim, on="device_name", how="left")
      .sort_values("Total", ascending=False)
)

# reorder columns: manufacturer BEFORE device_name
mno_cols = [c for c in final.columns if c not in {
    "manufacturer", "device_name", "device_type", "Technology", "SIM Slot", "Total"
}]
final = final[
    ["manufacturer", "device_name"]
    + mno_cols
    + ["Total", "device_type", "Technology", "SIM Slot"]
]

# ---- display
final.head(100)

,manufacturer,device_name,AIRTEL,MTN,Total,device_type,Technology,SIM Slot
33603,Itel Technology Limited,it2160,3480017,4289765,7769782,Mobile Phone/Feature phone,NONE,2
33690,Itel Technology Limited,itel it2163N,895792,793889,1689681,Mobile Phone/Feature phone,NONE,2
28706,Tecno Telecom (HK) Limited,TECNO T528,602459,972096,1574555,Mobile Phone/Feature phone,NONE,2
34896,NaN,NaN,679491,805570,1485061,NaN,NaN,NaN
28617,Tecno Telecom (HK) Limited,TECNO T101,564851,752450,1317301,Mobile Phone/Feature phone,NONE,2
...,...,...,...,...,...,...,...,...
28637,Tecno Telecom (HK) Limited,TECNO T313,22078,26900,48978,Mobile Phone/Feature phone,NONE,2
1846,Mobiwire Mobiles (Ningbo) Co Ltd,Alola 4G,12808,36063,48871,Smartphone,4G,2
3199,"Shenzhen Sprocomm Technologies Co., Ltd.",C22,20255,28362,48617,Smartphone,4G,2
7187,Samsung Korea,Galaxy A14,11910,35525,47435,Smartphone,4G,2


In [89]:
len(final)

34897

In [90]:
# Export to CSV (Downloads folder)
OUT_CSV = "/Users/wmuheki/Downloads/device_name_summ.csv"

final.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8"
)

print(f"Saved to {OUT_CSV}")

Saved to /Users/wmuheki/Downloads/device_name_summ.csv


In [99]:
import pandas as pd

df = df_domestic_devices_gold.copy()

# --- normalize
for c in ["manufacturer", "device_name", "imei"]:
    df[c] = df[c].astype(str).str.strip()

df["gender"] = (
    df.get("gender")
      .astype("string")
      .str.strip()
      .str.title()
      .fillna("UNKNOWN")
)
df.loc[df["gender"].isin(["", "<NA>", "Nan", "None"]), "gender"] = "UNKNOWN"

# --- 4G / 5G categorisation (4G, 5G, 4G+5G, NONE/UNKNOWN)
has4 = pd.to_numeric(df.get("has_4g"), errors="coerce").fillna(0).astype(int)
has5 = pd.to_numeric(df.get("has_5g"), errors="coerce").fillna(0).astype(int)

df["Technology"] = "UNKNOWN"
df.loc[(has4 == 1) & (has5 == 0), "Technology"] = "4G"
df.loc[(has4 == 0) & (has5 == 1), "Technology"] = "5G"
df.loc[(has4 == 1) & (has5 == 1), "Technology"] = "4G+5G"
df.loc[(has4 == 0) & (has5 == 0), "Technology"] = "NONE"

# --- pivot: (manufacturer, device_name, Technology) x gender
by_gender = df.pivot_table(
    index=["manufacturer", "device_name", "Technology"],
    columns="gender",
    values="imei",
    aggfunc=pd.Series.nunique,   # change to "size" for raw row counts
    fill_value=0,
    dropna=False
)

# --- total across gender columns
by_gender["Total"] = by_gender.sum(axis=1)

# --- sort + display
by_gender = by_gender.sort_values("Total", ascending=False).reset_index()
by_gender.head(100)

gender,manufacturer,device_name,Technology,Female,Male,UNKNOWN,Undefined,Total
0,Itel Technology Limited,it2160,NONE,3703691,3181068,1092245,74,7977078
1,Itel Technology Limited,itel it2163N,NONE,785192,680986,276777,16,1742971
2,Tecno Telecom (HK) Limited,TECNO T528,NONE,616174,763773,228820,14,1608781
3,NaN,NaN,NONE,650907,634046,237591,15,1522559
4,Tecno Telecom (HK) Limited,TECNO T101,NONE,604245,548219,198334,19,1350817
...,...,...,...,...,...,...,...,...
95,Itel Technology Limited,itel it5619,NONE,20133,23039,7396,0,50568
96,Tecno Telecom (HK) Limited,TECNO T313,NONE,20222,22002,7514,1,49739
97,Tecno Telecom (HK) Limited,TECNO T315,4G,18316,21512,8215,0,48043
98,Samsung Korea,Galaxy A14,4G,20724,18620,8513,1,47858


In [101]:
import pandas as pd

df = df_domestic_devices_gold.copy()

# --- normalize
for c in ["manufacturer", "device_name", "imei"]:
    df[c] = df[c].astype(str).str.strip()

# --- 4G / 5G categorisation
has4 = pd.to_numeric(df.get("has_4g"), errors="coerce").fillna(0).astype(int)
has5 = pd.to_numeric(df.get("has_5g"), errors="coerce").fillna(0).astype(int)

df["Technology"] = "UNKNOWN"
df.loc[(has4 == 1) & (has5 == 0), "Technology"] = "4G"
df.loc[(has4 == 0) & (has5 == 1), "Technology"] = "5G"
df.loc[(has4 == 1) & (has5 == 1), "Technology"] = "4G+5G"
df.loc[(has4 == 0) & (has5 == 0), "Technology"] = "NONE"

# --- age cleanup
age = pd.to_numeric(df.get("age"), errors="coerce")

# --- define buckets (18 to 99)
bins   = [18, 24, 34, 44, 54, 64, 74, 84, 99]
labels = ["18–24", "25–34", "35–44", "45–54", "55–64", "65–74", "75–84", "85–99"]

df["age_bracket"] = pd.cut(age, bins=bins, labels=labels, include_lowest=True)
df["age_bracket"] = df["age_bracket"].astype("string").fillna("UNKNOWN")

# keep ordering
df["age_bracket"] = pd.Categorical(
    df["age_bracket"],
    categories=labels + ["UNKNOWN"],
    ordered=True
)

# --- pivot: (manufacturer, device_name, Technology) x age_bracket
by_age = df.pivot_table(
    index=["manufacturer", "device_name", "Technology"],
    columns="age_bracket",
    values="imei",
    aggfunc=pd.Series.nunique,   # change to "size" for raw row counts
    fill_value=0,
    dropna=False
)

# --- total across brackets
by_age["Total"] = by_age.sum(axis=1)

# --- sort + display
by_age = by_age.sort_values("Total", ascending=False).reset_index()
by_age.head(100)

age_bracket,manufacturer,device_name,Technology,18–24,25–34,35–44,45–54,55–64,65–74,75–84,85–99,UNKNOWN,Total
0,Itel Technology Limited,it2160,NONE,433894,2352884,1860319,1286903,722959,298370,109130,27426,1093131,8185016
1,Itel Technology Limited,itel it2163N,NONE,133276,554025,379182,239086,128968,52922,18616,4699,276952,1787726
2,Tecno Telecom (HK) Limited,TECNO T528,NONE,56809,422488,382299,286620,171860,69903,24768,5640,228985,1649372
3,NaN,NaN,NONE,129673,513769,338933,209360,100962,36240,12329,3180,237719,1582165
4,Tecno Telecom (HK) Limited,TECNO T101,NONE,69436,386232,316865,213339,124364,50731,18347,4320,198488,1382122
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Itel Technology Limited,itel it5619,NONE,1895,13469,12447,8646,4789,1913,619,152,7399,51329
96,Tecno Telecom (HK) Limited,TECNO T313,NONE,2283,13324,11502,8121,4812,2153,777,170,7516,50658
97,Tecno Telecom (HK) Limited,TECNO T315,4G,2208,13292,11502,7480,4041,1522,491,108,8220,48864
98,Samsung Korea,Galaxy A14,4G,3171,15618,10496,6100,3026,1076,274,57,8517,48335


In [104]:
by_age = by_age[by_age["Technology"] != "NONE"].reset_index(drop=True)

by_age.head(100)

age_bracket,manufacturer,device_name,Technology,18–24,25–34,35–44,45–54,55–64,65–74,75–84,85–99,UNKNOWN,Total
0,Samsung Korea,Galaxy A05,4G,72741,316415,193161,107383,51035,18230,6010,1629,116133,882737
1,Tecno Telecom (HK) Limited,Spark Go 2024,4G,62154,241390,150129,91110,43363,14109,4416,1082,93851,701604
2,Samsung Korea,Galaxy A06,4G,36917,141193,86365,49106,23509,8305,2637,775,61163,409970
3,Tecno Telecom (HK) Limited,Pop 7,4G,24091,118130,82097,50316,24473,8071,2282,533,47235,357228
4,Samsung Korea,Galaxy A03 Core,4G,19059,98639,68453,37153,17246,5767,1808,428,38309,286862
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,ZTE Corporation,ZTE MF935N,4G,716,3064,2095,967,374,126,26,8,13123,20499
96,INFINIX TECHNOLOGY LIMITED,Smart 10 Plus,4G,1872,7035,4390,2333,1042,291,69,15,3369,20416
97,Xiaomi Communications Co Ltd,13C,4G,1552,6775,4280,2426,1246,374,94,16,3523,20286
98,Samsung Korea,Galaxy S10,4G,2487,8055,3611,2046,966,322,83,14,2700,20284


In [106]:
by_gender = by_gender[by_gender["Technology"] != "NONE"].reset_index(drop=True)

by_gender.head(100)

gender,manufacturer,device_name,Technology,Female,Male,UNKNOWN,Undefined,Total
0,Samsung Korea,Galaxy A05,4G,345345,401878,116068,10,863301
1,Tecno Telecom (HK) Limited,Spark Go 2024,4G,291431,304468,93799,3,689701
2,Samsung Korea,Galaxy A06,4G,152432,187910,61137,3,401482
3,Tecno Telecom (HK) Limited,Pop 7,4G,161465,142858,47218,3,351544
4,Samsung Korea,Galaxy A03 Core,4G,121570,121810,38289,2,281671
...,...,...,...,...,...,...,...,...
95,ZTE Corporation,ZTE MF935N,4G,2231,5085,13123,0,20439
96,INFINIX TECHNOLOGY LIMITED,Smart 10 Plus,4G,7032,9835,3369,0,20236
97,Xiaomi Communications Co Ltd,13C,4G,8473,8104,3521,0,20098
98,Samsung Korea,Galaxy S10,4G,7742,9574,2698,0,20014


In [107]:
len(by_gender)

222258993

In [ ]:

# Export to CSV (Downloads folder)
OUT_CSV = "/Users/wmuheki/Downloads/device_age.csv"

by_age.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8"
)

print(f"Saved to {OUT_CSV}")

In [95]:
qwref

NameError: name 'qwref' is not defined

In [63]:
all_devices_gold = (
    df_domestic_devices_gold
    .pivot_table(
        index="device_type",   # rows
        columns="mno",         # columns
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

# Row-wise total (across MNOs)
all_devices_gold["Total"] = all_devices_gold.sum(axis=1)

# Optional: sort by Total descending
all_devices_gold = all_devices_gold.sort_values("Total", ascending=False)

all_devices_gold.head(100)

mno,AIRTEL,MTN,Total
device_type,,,
Mobile Phone/Feature phone,12669684,14147322,26817006
Smartphone,5521500,8930264,14451764
<NA>,817573,979491,1797064
Modem,58860,339763,398623
Handheld,150623,188791,339414
Tablet,41379,111987,153366
WLAN Router,17336,109007,126343
IoT Device,36299,32423,68722
Dongle,43743,22485,66228


In [64]:
all_man_gold = (
    df_domestic_devices_gold
    .pivot_table(
        index="manufacturer",   # rows
        columns="mno",         # columns
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

# Row-wise total (across MNOs)
all_man_gold["Total"] = all_man_gold.sum(axis=1)

# Optional: sort by Total descending
all_man_gold = all_man_gold.sort_values("Total", ascending=False)

all_man_gold.head(100)

mno,AIRTEL,MTN,Total
manufacturer,,,
Itel Technology Limited,8797565,9712197,18509762
Tecno Telecom (HK) Limited,3998547,5764753,9763300
Samsung Korea,1191009,2424843,3615852
<NA>,817573,979491,1797064
INFINIX TECHNOLOGY LIMITED,653006,1038241,1691247
...,...,...,...
Shenzhen Jimi Iot Co Ltd,3547,3215,6762
S.H.X Mobile Phone Trading L.L.C,3320,3435,6755
Mobitel Ltd,3344,3401,6745


In [65]:
all_tac_gold = (
    df_domestic_devices_gold
    .pivot_table(
        index="mno",
        columns="tac_status",
        values="imei",          # any column; we're just counting rows
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

all_tac_gold["Total"] = all_tac_gold.sum(axis=1)

# optional: sort by Total desc
all_tac_gold = all_tac_gold.sort_values("Total", ascending=False)

display(all_tac_gold)

tac_status,FAKE,VALID,Total
mno,,,
MTN,979491,23914917,24894408
AIRTEL,817573,18565735,19383308


In [ ]:
all_tac_gold.head()

## For all devices seen in last 90 days (multiple device usage) - CEIR

#### Only Smartphones

In [ ]:
tab_type = (
    df_domestic_devices_bronze
    .loc[df_domestic_devices_bronze["device_type"].str.lower() == "smartphone"]
    .pivot_table(
        index="device_name",   # rows
        columns="mno",          # columns
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

# Row-wise total (across MNOs)
tab_type["Total"] = tab_type.sum(axis=1)

# Optional: sort by Total descending
tab_type = tab_type.sort_values("Total", ascending=False)

tab_type.head(100)

In [ ]:
tab_type = (
    df_domestic_devices_bronze
    .loc[df_domestic_devices_bronze["device_type"].str.lower() == "smartphone"]
    .pivot_table(
        index="manufacturer",   # rows
        columns="mno",          # columns
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

# Row-wise total (across MNOs)
tab_type["Total"] = tab_type.sum(axis=1)

# Optional: sort by Total descending
tab_type = tab_type.sort_values("Total", ascending=False)

tab_type.head(50)

In [ ]:
tab_tac = (
    df_domestic_devices_bronze
    .loc[df_domestic_devices_bronze["device_type"].str.lower() == "smartphone"]
    .pivot_table(
        index="mno",
        columns="tac_status",
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

tab_tac["Total"] = tab_tac.sum(axis=1)

# Optional: sort by Total descending
tab_tac = tab_tac.sort_values("Total", ascending=False)

tab_tac.head()

### All Device Types

In [ ]:
tab_type = (
    df_domestic_devices_bronze
    .pivot_table(
        index="device_type",   # rows
        columns="mno",         # columns
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

# Row-wise total (across MNOs)
tab_type["Total"] = tab_type.sum(axis=1)

# Optional: sort by Total descending
tab_type = tab_type.sort_values("Total", ascending=False)

tab_type.head(50)

In [ ]:
tab_type = (
    df_domestic_devices_bronze
    .pivot_table(
        index="manufacturer",   # rows
        columns="mno",         # columns
        values="imei",
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

# Row-wise total (across MNOs)
tab_type["Total"] = tab_type.sum(axis=1)

# Optional: sort by Total descending
tab_type = tab_type.sort_values("Total", ascending=False)

tab_type.head(100)

In [ ]:
tab_tac = (
    df_domestic_devices_bronze
    .pivot_table(
        index="mno",
        columns="tac_status",
        values="imei",          # any column; we're just counting rows
        aggfunc="size",
        fill_value=0,
        dropna=False
    )
)

tab_tac["Total"] = tab_tac.sum(axis=1)

# optional: sort by Total desc
tab_tac = tab_tac.sort_values("Total", ascending=False)

tab_tac.head()

In [ ]:
df_domestic_devices_bronze.head()

In [ ]:
"""
IMEIs used by the same MSISDNs - grouped by TAC Status
"""

imei_per_msisdn = (
    df_domestic_devices_bronze
    .groupby(["msisdn", "tac_status"])["imei"]
    .nunique()
    .reset_index(name="imei_count")
)

imei_per_msisdn.head()

In [ ]:

"""
 Identify duplicate records (core check)
 """

dup_mask = df_domestic_devices_bronze.duplicated(
    subset=["msisdn", "imsi", "imei"],
    keep=False   # marks ALL duplicates, not just later ones
)

duplicates = df_domestic_devices_bronze.loc[
    dup_mask,
    ["msisdn", "imsi", "imei", "tac", "device_name", "tac_status"]
]

duplicates.head(20)

In [ ]:

"""
Count IMEI frequency by MNO × TAC status - Top 100
"""
imei_freq = (
    df_domestic_devices_bronze
    .groupby(["mno", "tac_status", "imei"])
    .size()
    .reset_index(name="records")
)


TOP_N = 100

top_imei = (
    imei_freq
    .sort_values(["mno", "tac_status", "records"], ascending=[True, True, False])
    .groupby(["mno", "tac_status"])
    .head(TOP_N)
)

top_imei.head(100)

## use KYC data to estimate actual Airtel Subscribers

In [ ]:
df_NID.head()

In [ ]:
df_NID.head()

In [ ]:
"""


# Build a unique MSISDN → demographics map
nin_map = (
    df_NID[["msisdn", "gender", "age"]]
    .astype({"msisdn": "string"})
    .dropna(subset=["msisdn"])
    .drop_duplicates(subset=["msisdn"], keep="first")
)

# 1) Calls table (different key names → two MSISDN columns → drop right one)
df_calls_devices = (
    df_calls_devices
    .merge(
        nin_map,
        how="left",
        left_on="calling_number",
        right_on="msisdn"
    )
    .drop(columns=["msisdn"])
)

# 2) Domestic table (same key name → single MSISDN column)
df_ceir_domestic = df_ceir_domestic.merge(
    nin_map,
    how="left",
    on="msisdn"
)

# 3) Roamers table (same key name → single MSISDN column)
df_ceir_roamers = df_ceir_roamers.merge(
    nin_map,
    how="left",
    on="msisdn"
)



"""

In [110]:
import pandas as pd

FILE = "/Users/wmuheki/Downloads/Newest_file.csv"

device_prices = pd.read_csv(
    FILE,
    skiprows=4,          # skip title / notes / blank rows
    dtype=str,           # read everything safely first
    low_memory=False
)

# ---- clean column names
device_prices.columns = (
    device_prices.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[()]", "", regex=True)
)

device_prices.head()

,manufacturer,device_name,total,device_type,technology,sim_slots,estimated_min_cost_usd,estimated_max_cost_usd,average_cost_usd,estimated_tax_usd,markup_20%,estimated_final_price_ugx
0,Samsung Korea,Galaxy A05,"845,031",Smartphone,4G,2,95,110,103,38,28,"621,667"
1,Tecno Telecom (HK) Limited,Spark Go 2024,"680,655",Smartphone,4G,2,70,85,78,28,21,"470,041"
2,Samsung Korea,Galaxy A06,"394,806",Smartphone,4G,2,100,115,108,39,29,"651,992"
3,Tecno Telecom (HK) Limited,Pop 7,"348,306",Smartphone,4G,2,60,70,65,24,18,"394,228"
4,Samsung Korea,Galaxy A03 Core,"277,838",Smartphone,4G,2,55,65,60,22,16,"363,902"


In [112]:
device_prices.describe

<bound method NDFrame.describe of                           manufacturer      device_name       total  \
0                        Samsung Korea       Galaxy A05     845,031   
1           Tecno Telecom (HK) Limited    Spark Go 2024     680,655   
2                        Samsung Korea       Galaxy A06     394,806   
3           Tecno Telecom (HK) Limited            Pop 7     348,306   
4                        Samsung Korea  Galaxy A03 Core     277,838   
...                                ...              ...         ...   
5089  vivo Mobile Communication Co Ltd      Z10 Lite 5G          10   
5090                         Not Known      Ascend Y550          10   
5091        HUAWEI Technologies Co Ltd        Y3 (2017)          10   
5092               Glory Prize Limited        I KALL Z6          10   
5093                               NaN              NaN  11,745,930   

     device_type technology sim_slots estimated_min_cost_usd  \
0     Smartphone         4G         2            

In [113]:
device_prices["average_cost_usd"].mode()

0    0
Name: average_cost_usd, dtype: str

In [120]:
device_prices = device_prices[device_prices["estimated_final_price_ugx"] != 0].reset_index(drop=True)

In [130]:
device_prices = device_prices[
    device_prices["estimated_final_price_ugx"].notna()
].reset_index(drop=True)

In [131]:
device_prices["estimated_final_price_ugx"].count()

np.int64(1674)

In [132]:
len(device_prices)

1674

In [116]:
device_prices.head(100)

,manufacturer,device_name,total,device_type,technology,sim_slots,estimated_min_cost_usd,estimated_max_cost_usd,average_cost_usd,estimated_tax_usd,markup_20%,estimated_final_price_ugx
0,Samsung Korea,Galaxy A05,"845,031",Smartphone,4G,2,95,110,103,38,28,"621,667"
1,Tecno Telecom (HK) Limited,Spark Go 2024,"680,655",Smartphone,4G,2,70,85,78,28,21,"470,041"
2,Samsung Korea,Galaxy A06,"394,806",Smartphone,4G,2,100,115,108,39,29,"651,992"
3,Tecno Telecom (HK) Limited,Pop 7,"348,306",Smartphone,4G,2,60,70,65,24,18,"394,228"
4,Samsung Korea,Galaxy A03 Core,"277,838",Smartphone,4G,2,55,65,60,22,16,"363,902"
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Tecno Telecom (HK) Limited,Spark 5 Air,"21,248",Smartphone,4G,2,60,75,68,25,18,"409,390"
96,Apple Inc,iPhone 12 Pro Max,"20,903",Smartphone,4G+5G,1,450,600,525,192,143,"3,184,146"
97,Samsung Korea,Galaxy A23,"20,589",Smartphone,4G,2,150,180,165,60,45,"1,000,732"
98,Tecno Telecom (HK) Limited,Spark 9,"20,268",Smartphone,4G,2,25,45,35,13,10,"212,276"


0    140
Name: average_cost_usd, dtype: str